# CRISP-DM Fase 3: Data Preparation - Limpieza, Wrangling y Gobernanza PII
## Dataset: SIPSA Abastecimiento de Alimentos (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 3: Data Cleaning & Transformation) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: DAMA-DMBOK 2 (Data Security & Governance), Ley 1581 de 2012 (Habeas Data Colombia), PEP 8.

---

### 1. Gobernanza de Datos y Protección de la Privacidad (Ley 1581)
En cumplimiento de la legislación colombiana sobre protección de datos personales (**Ley Estatutaria 1581 de 2012**) y los estándares de **DAMA-DMBOK 2**:
* Cualquier dato personal identificable (**PII** - *Personally Identifiable Information*), tales como nombres de agricultores, cédulas, números de teléfono o correos electrónicos, debe ser **anonimizado de forma irreversible** mediante algoritmos de dispersión criptográfica unidireccional (**SHA-256**).
* La anonimización preserva la capacidad analítica de correlacionar transacciones sin comprometer la identidad legal de los titulares.

### 2. Estandarización a Formato snake_case
Todos los nombres de columnas son normalizados a minúsculas, sustituyendo espacios y caracteres especiales por guiones bajos (`_`), eliminando tildes y diacríticos conforme a las directrices de código limpio de **PEP 8**.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 3. Pipeline de Sanitización y Anonimización
Aplicamos `DataSanitizer` para normalizar cadenas y `PIIHandler` para aplicar hashing SHA-256 a columnas sensibles si están presentes.


In [ ]:
# ==============================================================================
# [FASE 5: NORMALIZACIÓN SNAKE_CASE Y ANONIMIZACIÓN CRIPTOGRÁFICA PII]
# ==============================================================================
from src.cleaning.sanitizer import DataSanitizer
from src.cleaning.pii_handler import PIIHandler

parquet_file = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
df = pd.read_parquet(parquet_file)

print(f"📦 Columnas antes de normalización: {list(df.columns)[:5]}...")

# 1. Normalización de identificadores a snake_case
df_clean = DataSanitizer.clean_column_names(df)

# 2. Sanitización de cadenas (eliminación de espacios en blanco y caracteres de control)
df_clean = DataSanitizer.sanitize_strings(df_clean)

# 3. Detección y anonimización de PII (Ley 1581)
pii_columns = [col for col in df_clean.columns if any(p in col.lower() for p in ['email', 'correo', 'telefono', 'nombre', 'contacto', 'identificacion'])]
if pii_columns:
    print(f"🔒 Campos de datos personales sensibles detectados: {pii_columns} -> Aplicando SHA-256")
    for pii_col in pii_columns:
        df_clean[pii_col] = PIIHandler.anonymize_series(df_clean[pii_col])
else:
    print("ℹ️ No se identificaron campos PII sensibles en el dataset. Cumplimiento legal verificado.")

print(f"✅ DataFrame sanitizado exitosamente. Dimensiones: {df_clean.shape}")


### 4. Persistencia en la Capa Silver (Parquet con Compresión Snappy)
Los datos limpios se guardan en formato **Parquet** utilizando pyarrow.
* **Almacenamiento Columnar**: Permite escanear únicamente las columnas consultadas, reduciendo el I/O en órdenes de magnitud.
* **Compresión Snappy**: Provee un balance óptimo entre ratio de compresión (reducción de hasta un 70% del tamaño original) y velocidad de descompresión en memoria.


In [ ]:
# ==============================================================================
# [FASE 5: EXPORTACIÓN A CAPA SILVER EN FORMATO PARQUET SNAPPY]
# ==============================================================================
out_parquet = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
out_parquet.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(out_parquet, engine="pyarrow", compression="snappy", index=False)
file_size_kb = out_parquet.stat().st_size / 1024

print(f"💾 Archivo persistido en Capa Silver:")
print(f"• Destino: {out_parquet.relative_to(APP_ROOT)}")
print(f"• Tamaño optimizado en disco: {file_size_kb:.2f} KB")
print(f"• Total de registros curados: {len(df_clean):,} filas")


### 5. Conclusión de Limpieza y Gobernanza
* **Resultado**: Capa Silver consolidada, normalizada, libre de datos sensibles desprotegidos y optimizada para lectura columnar.
* **Siguiente Paso**: Continuar a `06_modelo_base_de_datos.ipynb` para diseñar el modelo dimensional e integrar la tabla en el Data Lakehouse SQL.
